# Big Data Platforms — MapReduce & Apache Spark

**A study notebook based on Lecture 2 of *Big Data Platforms* (DATA140031 / DATA140032), University of Helsinki, by Keijo Heljanko.**

> This notebook is an expanded, self-contained, runnable companion to the original slide deck (`lecture2.pdf`, included in this repo under `slides/`). Every idea from the original lecture is preserved, but each slide has been turned into a fuller explanation, with extra context, worked examples, and — where possible — **actual runnable code** instead of just bullet points.

**How to use this notebook**
- Read top to bottom — sections build on each other, exactly like the lecture.
- Code cells are real, executable PySpark / Python. If you have Java + `pyspark` installed (`pip install pyspark`), you can re-run every cell yourself.
- Diagrams are the original figures from the slides, extracted as images so nothing visual is lost.

**Contents**
1. [Why Big Data needs a special kind of platform](#1)
2. [Google MapReduce](#2)
3. [Why MapReduce forbids side effects](#3)
4. [Worked example: Word Count by hand](#4)
5. [Apache Spark — motivation](#5)
6. [Resilient Distributed Datasets (RDDs)](#6)
7. [Hands-on: PySpark Word Count](#7)
8. [RDD Transformations & Actions — reference](#8)
9. [Lineage: narrow vs. wide dependencies](#9)
10. [DAG scheduling & pipelining into stages](#10)
11. [Spark ecosystem extensions](#11)
12. [Summary & further reading](#12)


<a id="1"></a>
## 1. Why Big Data needs a special kind of platform

When people say "Big Data" they usually mean data that is too large, arrives too fast, or is too varied to process on a single machine — think of Google trying to index a meaningful fraction of the entire web, or a video-sharing site storing petabytes of user uploads.

The original lecture frames the core engineering problem like this:

> *"When dealing with Big Data (a substantial portion of the Internet in the case of Google!), the only viable option is to use servers and hard disks **in parallel**."*

That single sentence hides a lot of complexity. If you naively spread work across, say, 1,000 machines, you now also have to deal with:

- **Parallelization** — splitting the work into independent pieces.
- **Synchronization** — making sure pieces that depend on each other wait for one another correctly.
- **Load balancing** — making sure no single machine becomes the bottleneck.
- **Fault tolerance** — with 1,000 machines running for hours, *something* will fail (disk, network, power) before the job finishes. The system must recover automatically, without corrupting the result.

Doing all of this by hand, in every program you write, is a nightmare. The insight behind **Google MapReduce** (and later **Apache Spark**) is: *hide all of this behind a very restricted programming model*, so the programmer only writes simple, sequential-looking functions, and the framework takes care of the rest.


<a id="2"></a>
## 2. Google MapReduce

**MapReduce** is a scalable batch-processing framework, originally developed at Google to build their web index — years before Apache Spark existed. It is designed for:

- Hundreds to thousands of machines working in parallel.
- Job runtimes from minutes to hours (i.e. *batch* processing, not real-time).
- Maximum programmer productivity — nobody should have to reinvent disk-parallelism tooling for every new job.

**Reference paper:** J. Dean and S. Ghemawat, *"MapReduce: Simplified Data Processing on Large Clusters"*, OSDI 2004.

### The programming model

MapReduce is based on **functional programming "in the large"**. The user is only allowed to write two side-effect-free functions:

| Function | Role |
|---|---|
| **Map** | Takes one input record, produces zero or more `(key, value)` pairs. |
| **Reduce** | Takes *all* the values associated with one key, and combines them into a result. |

Two classic functional-programming ideas underpin this:

**Mapping a list** — apply a function to every element independently (so it's trivially parallelizable):

![Mapping a list with a map function](images/mapping_a_list.png)

**Reducing a list** — combine all elements of a list into a single output value using a folding/aggregating function:

![Reducing a list with a reduce function](images/reducing_a_list.png)

### Grouping map output by key (the "Shuffle")

The Map function emits `(key, value)` pairs. Before Reduce can run, the framework must **group all values that share the same key** together — this is the *Shuffle* phase. Crucially, each key's group of values can be reduced **independently and in parallel**:

![Keys divide map output to reducers](images/grouping_by_key.png)

### The full data flow

Practical MapReduce systems split the input into large blocks (64 MB+), each fed to a Map task. Intermediate `(key, value)` pairs are exchanged between nodes by the shuffle process, then reduced locally on each node:

![High level MapReduce dataflow](images/mapreduce_dataflow.png)

### End-to-end system diagram

The classic diagram from the original Dean & Ghemawat (2004) paper shows the full lifecycle: the user program forks a `Master` and several `Worker` processes; the master assigns Map and Reduce tasks to idle workers; map workers read input splits, write intermediate data to local disk; reduce workers remote-read the intermediate data relevant to their key range and write the final output files.

![MapReduce system diagram (Dean & Ghemawat, OSDI 2004)](images/mapreduce_diagram.png)

**Recap, in one paragraph:** the framework only lets you write a `Map` and a `Reduce` function. Map is fed 64–128 MB blocks and emits `(key, value)` pairs. The framework groups (shuffles) all values sharing a key into `(key, [list of values])`, which is fed to Reduce. A **Master** process schedules all of this and re-executes any Map or Reduce task whose worker machine fails.

**Apache Hadoop** is the best-known open-source implementation of MapReduce, historically used at scale by Yahoo!, Facebook, and Twitter.


<a id="3"></a>
## 3. Why MapReduce forbids side effects

This is one of the more subtle — and important — design decisions in the lecture, so it's worth slowing down on.

**Fault tolerance via re-execution.** If a machine running a Map or Reduce task dies mid-job (hardware failure, network partition, etc.), MapReduce's answer is refreshingly simple: **just re-run that task on a different machine**. No checkpointing of partial in-memory state, no complex recovery protocol.

**But this only works if the task is a pure function.** If your Map or Reduce function has *side effects* (e.g. it writes to an external database, increments a global counter, or depends on mutable shared state), then:

- Re-executing it on another machine could **duplicate** those side effects (the DB write happens twice).
- You cannot faithfully **re-create the environment** the original task ran in, since part of that environment was mutated and is now gone.

So the framework enforces (well — *strongly recommends*, since it can't actually stop you from writing impure code) that Map and Reduce are **side-effect-free**. In return you get two nice properties:

1. **Determinism** — the same input always produces the same output, no matter how many machines you run on.
2. **Debuggability** — you can run the exact same code on *one* machine (e.g. your laptop) to debug it, and it behaves identically to the 1,000-machine cluster run.

> *"It is easy to introduce side-effects to MapReduce programs as the framework does not enforce a strict programming methodology. However, the behavior of such programs is undefined by the framework, and should therefore be avoided."*

The only state that genuinely needs to survive failures is the **input/output data itself**, which is why MapReduce relies on a **fault-tolerant distributed filesystem** (like HDFS) to store job inputs and outputs — the compute is stateless and disposable; the storage layer is what's actually resilient.


<a id="4"></a>
## 4. Worked example: Word Count, done by hand

Word Count is the "Hello World" of MapReduce. We'll first do it with **plain Python**, simulating exactly what a MapReduce cluster would do internally: a Map phase, a Shuffle phase, and a Reduce phase — as three clearly separated steps. This builds the mental model before we let Spark do it for us in Section 7.

**Input file** (the classic example from the [Hadoop MapReduce tutorial](https://hadoop.apache.org/docs/current1/mapred_tutorial.html)):

```
Hello World Bye World
Hello Hadoop Goodbye Hadoop
```


In [1]:
# The raw input, as if read line-by-line from a distributed file
input_lines = [
    "Hello World Bye World",
    "Hello Hadoop Goodbye Hadoop",
]
input_lines

['Hello World Bye World', 'Hello Hadoop Goodbye Hadoop']

### Step 1 — Map

Each line is fed to the Map function, which emits `(word, 1)` for every word it sees.
In a real cluster, each line (or block of lines) would be processed on a different machine, completely independently.

In [2]:
def map_function(line):
    """Emits (word, 1) for every word in the line."""
    return [(word, 1) for word in line.split(" ")]

# Simulate running map() independently over every line (order across
# lines is not guaranteed on a real cluster -- that's fine, Map has no
# side effects and doesn't need to know about other lines).
map_output = []
for line in input_lines:
    map_output.extend(map_function(line))

map_output

[('Hello', 1),
 ('World', 1),
 ('Bye', 1),
 ('World', 1),
 ('Hello', 1),
 ('Hadoop', 1),
 ('Goodbye', 1),
 ('Hadoop', 1)]

### Step 2 — Shuffle (group by key)

The framework now groups every `(key, value)` pair by key, producing `(key, [list of values])`. This is the one step that requires moving data between machines over the network — everything before and after it can happen locally.

In [3]:
from collections import defaultdict

def shuffle(pairs):
    """Groups (key, value) pairs into {key: [values...]}."""
    grouped = defaultdict(list)
    for key, value in pairs:
        grouped[key].append(value)
    return dict(grouped)

shuffled = shuffle(map_output)
# Print in sorted order just for readability -- MapReduce doesn't
# guarantee an order here.
for k in sorted(shuffled):
    print(f"({k!r}, {shuffled[k]})")

('Bye', [1])
('Goodbye', [1])
('Hadoop', [1, 1])
('Hello', [1, 1])
('World', [1, 1])


### Step 3 — Reduce

Each `(key, [values])` group is independently reduced — in our case, by summing the list — to produce the final `(word, count)` pairs. On a real cluster, different keys' reductions can run on completely different machines in parallel, since summation is **associative and commutative** (more on why that matters in Section 8.3).

In [4]:
def reduce_function(key, values):
    """Sums the list of 1s for a given word."""
    return (key, sum(values))

final_counts = dict(reduce_function(k, v) for k, v in shuffled.items())

for word in sorted(final_counts):
    print(f"({word!r}, {final_counts[word]})")

('Bye', 1)
('Goodbye', 1)
('Hadoop', 2)
('Hello', 2)
('World', 2)


This should match the lecture's worked example exactly:

```
(Bye, 1)
(Goodbye, 1)
(Hadoop, 2)
(Hello, 2)
(World, 2)
```

That's the entire MapReduce model. Everything Spark adds on top (Section 5 onward) is about doing this *faster* and with a *richer* set of operations — the fundamental Map → Shuffle → Reduce shape stays the same.


<a id="5"></a>
## 5. Apache Spark — motivation

**Apache Spark** is a distributed programming framework for Big Data processing, based on functional programming, that **extends** the Google MapReduce model rather than replacing its core ideas. It offers Scala-collections-like interfaces for **Scala, Java, Python (PySpark), and R**.

**Original paper:** Zaharia, Chowdhury, Das, Dave, Ma, McCauly, Franklin, Shenker, Stoica — *"Resilient Distributed Datasets: A Fault-Tolerant Abstraction for In-Memory Cluster Computing"*, NSDI 2012.

### What problem does Spark solve?

MapReduce was tailored for batch jobs where data is read from and written back to disk (HDFS) at every stage. That's fine for a single pass over the data, but it's wasteful for:

- **Iterative algorithms** — e.g. most machine-learning training loops (gradient descent, PageRank, k-means) re-read the same dataset dozens or hundreds of times. Re-reading from disk every iteration is extremely slow.
- **Interactive / exploratory analysis** — a data scientist running many small queries against the same dataset doesn't want to pay the disk I/O cost every single time.

Spark's core idea: **keep intermediate data in RAM** across operations (when it fits), instead of always round-tripping through disk like classic MapReduce.

### Benchmark claims — and a reality check

The original Spark paper claims:

- **Up to 100× faster** than MapReduce when data fits in RAM.
- **Up to 10× faster** when data must spill to disk.

Independent, more rigorous benchmarking paints a more nuanced picture:

> Juwei Shi, Yunjie Qiu, Umar Farooq Minhas, Limei Jiao, Chen Wang, Berthold Reinwald, Fatma Özcan — *"Clash of the Titans: MapReduce vs. Spark for Large Scale Data Analytics"*, PVLDB 8(13): 2110-2121 (2015). [Paper (PDF)](http://www.vldb.org/pvldb/vol8/p2110-shi.pdf)

When **both** frameworks are constrained to use hard disks (i.e. not the favourable RAM-vs-disk comparison), Spark is roughly **2.5–5× faster for CPU-bound workloads** — a real but much more modest win — while MapReduce could still **out-sort** Spark on some disk-bound sorting workloads.

**Takeaway:** Spark's biggest wins come specifically from *iterative* and *in-memory* workloads. For a single disk-to-disk batch pass, the gap is much smaller than the headline "100x" number suggests.


<a id="6"></a>
## 6. Resilient Distributed Datasets (RDDs)

The core abstraction in Spark is the **RDD — Resilient Distributed Dataset**: a distributed data structure that lets processed data live spread across many machines' memory (or disk), while still looking, from the programmer's point of view, roughly like a single (very large) collection.

Key properties:

- **Partitioned.** The framework stores an RDD as a set of **partitions**. Each computer can hold several partitions, and each partition can be processed by its own thread — this is Spark's basic unit of parallelism.
- **Immutable.** Once created, an RDD's contents never change. Every "transformation" produces a *new* RDD rather than mutating the old one. (This mirrors why MapReduce forbids side effects — see Section 3.)
- **Resilient via lineage, not replication.** Instead of replicating data for fault tolerance (expensive!), each RDD partition remembers its **lineage**: the recipe (parent RDDs + the operation applied) needed to recompute it from scratch. If a partition is lost, Spark just re-runs the recipe on another machine — the same re-execution idea as MapReduce, just applied at finer granularity.

### Where do RDDs come from?

1. **Local data structures or local files** — e.g. `sc.parallelize([...])` or a file on your own disk.
2. **Other RDDs, via transformations** — e.g. `.map()`, `.filter()`.
3. **Distributed storage systems** — this is the normal case for actual Big Data: Hadoop Distributed Filesystem (HDFS), Amazon S3, HBase, etc.

### Performance tuning: how many partitions?

Two competing forces when Spark first splits an RDD into partitions:

- **More partitions → better load balancing.** Rule of thumb: aim for **2–10× the total number of CPU cores** across your cluster, so that if one partition finishes early, there's more work ready to steal.
- **But partitions shouldn't be too small.** Each partition should take **more than ~100 ms** to process — otherwise the *overhead* of starting and finishing a task swamps the actual work being done.


<a id="7"></a>
## 7. Hands-on: PySpark Word Count

Now let's do the exact same Word Count job as Section 4, but for real, using **PySpark** and RDDs. This reproduces the lecture's worked example (which used the Finnish national epic *Kalevala* as input — see [Project Gutenberg #7000](http://www.gutenberg.org/ebooks/7000)) on a small sample text, so it runs instantly and reproducibly anywhere.

**To run this section yourself:**
```bash
pip install pyspark          # needs a local Java runtime (JDK 8+)
jupyter notebook              # then run the cells below
```


In [5]:
from pyspark import SparkContext, SparkConf

# local[*] tells Spark to run locally using all available CPU cores --
# no cluster needed for this notebook.
conf = SparkConf().setAppName("WordCountDemo").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("ERROR")  # quiet down Spark's own logging for this notebook

print("Spark version:", sc.version)

Spark version: 4.2.0


In [6]:
# Write out the same sample text used in Section 4, so PySpark can
# read it from "disk" just like a real job would.
with open("sample.txt", "w") as f:
    f.write("Hello World Bye World\n")
    f.write("Hello Hadoop Goodbye Hadoop\n")

with open("sample.txt") as f:
    print(f.read())

Hello World Bye World
Hello Hadoop Goodbye Hadoop



### Step 1 — read the file into an RDD

`sc.textFile(...)` creates an RDD where **each line is one element**. `.take(n)` is a debugging action that pulls the first `n` elements back to the driver — never use it (or `.collect()`) on a truly huge RDD, since it would try to bring all the data to a single machine!

In [7]:
lines_rdd = sc.textFile("sample.txt")
lines_rdd.take(10)

['Hello World Bye World', 'Hello Hadoop Goodbye Hadoop']

### Step 2 — split each line into words with `flatMap`

`flatMap` is like `map`, except each input element can produce **any number** of output elements (here: the words in that line), which are then flattened into one RDD — as opposed to `map`, which would give you a list-of-lists.

In [8]:
words_rdd = lines_rdd.flatMap(lambda line: line.split(" "))
words_rdd.collect()

['Hello', 'World', 'Bye', 'World', 'Hello', 'Hadoop', 'Goodbye', 'Hadoop']

### Step 3 — map each word to `(word, 1)`

Note: `map` (and `flatMap`) are pure functions with no side effects, so Spark is free to run them on any partition, on any machine, in any order.

In [9]:
count_rdd = words_rdd.map(lambda word: (word, 1))
count_rdd.collect()

[('Hello', 1),
 ('World', 1),
 ('Bye', 1),
 ('World', 1),
 ('Hello', 1),
 ('Hadoop', 1),
 ('Goodbye', 1),
 ('Hadoop', 1)]

### Step 4 — combine counts with `reduceByKey`

`reduceByKey` is Spark's version of the Shuffle+Reduce steps combined: it groups values by key and reduces each group with the function you give it — here, plain addition.

Under the hood, Spark doesn't naively shuffle every single `(word, 1)` pair across the network. Instead **each partition first computes its own local word counts**, and only the *per-partition totals* are shuffled to the final reducer — a huge bandwidth saving. This optimisation is only correct because `+` is **associative and commutative**, so it doesn't matter in what order or grouping the partial sums are combined.

In [10]:
totals_rdd = count_rdd.reduceByKey(lambda a, b: a + b)
sorted(totals_rdd.collect())

[('Bye', 1), ('Goodbye', 1), ('Hadoop', 2), ('Hello', 2), ('World', 2)]

This matches Section 4's hand-rolled Map/Shuffle/Reduce result exactly — as it must, since it's the same algorithm, just executed by Spark instead of by us.

Finally, always remember to release the Spark context when you're done (especially important in scripts; less critical in a long-lived notebook session):

In [11]:
sc.stop()

<a id="8"></a>
## 8. RDD Transformations & Actions — reference

Spark operations on RDDs come in two flavours, and the distinction matters a lot:

- **Transformations** — build a new RDD from an existing one (e.g. `map`, `filter`). They are **lazy**: Spark just records *what* you asked for (adding to a dependency graph), without doing any actual computation yet.
- **Actions** — trigger the actual computation and return a result to the driver program, or write output somewhere (e.g. `collect`, `count`, `saveAsTextFile`).

This laziness lets Spark look at your *entire* chain of transformations before running anything, and optimise the whole pipeline — e.g. by fusing several `map`/`filter` steps into a single pass over the data (see Section 10).

### Transformations

| Transformation | What it does |
|---|---|
| `map(func)` | Apply `func` to every element, 1-to-1. |
| `filter(func)` | Keep only elements where `func` returns true. |
| `flatMap(func)` | Like `map`, but each input can produce 0..N outputs, flattened. |
| `mapPartitions(func)` | Like `map`, but `func` gets a whole partition (an iterator) at once — useful for per-partition setup cost (e.g. opening a DB connection once). |
| `mapPartitionsWithIndex(func)` | Same, plus the partition's index number. |
| `sample(withReplacement, fraction, seed)` | Randomly sample a fraction of the data. |
| `union(other)` | Concatenate two RDDs. |
| `intersection(other)` | Elements present in both RDDs. |
| `distinct([numTasks])` | Remove duplicate elements. |
| `groupByKey([numTasks])` | Group values by key — *warning:* shuffles all raw values, no local pre-combination like `reduceByKey`. |
| `reduceByKey(func, [numTasks])` | Group by key **and** combine with `func`, pre-aggregating locally per partition first. |
| `aggregateByKey(zeroValue)(seqOp, combOp, [numTasks])` | Like `reduceByKey` but input/output types can differ. |
| `sortByKey([ascending], [numTasks])` | Sort an RDD of `(key, value)` pairs by key. |
| `join(other, [numTasks])` | Inner join two `(key, value)` RDDs on key. |
| `cogroup(other, [numTasks])` | Group values from two RDDs that share a key. |
| `cartesian(other)` | All pairs `(a, b)` — one from each RDD. |
| `pipe(command, [envVars])` | Pipe each partition through an external shell command. |
| `coalesce(numPartitions)` | Shrink the number of partitions (cheap, avoids a full shuffle where possible). |
| `repartition(numPartitions)` | Reshuffle into a given number of partitions (can grow or shrink; always a full shuffle). |
| `repartitionAndSortWithinPartitions(partitioner)` | Repartition and sort within each resulting partition in one shuffle. |

### Actions

| Action | What it does |
|---|---|
| `reduce(func)` | Combine all elements into one value with `func`. |
| `collect()` | Return **all** elements to the driver as a local list — only for small results! |
| `count()` | Number of elements. |
| `first()` | The first element. |
| `take(n)` | The first `n` elements. |
| `takeSample(withReplacement, num, [seed])` | A random sample of `num` elements, returned locally. |
| `takeOrdered(n, [ordering])` | The `n` smallest (or custom-ordered) elements. |
| `saveAsTextFile(path)` | Write the RDD as text file(s). |
| `saveAsSequenceFile(path)` | Write as a Hadoop SequenceFile. |
| `saveAsObjectFile(path)` | Write as serialized Java objects. |
| `countByKey()` | Count of elements per key, returned as a local dictionary/map. |
| `foreach(func)` | Run `func` on every element, for its side effects (e.g. writing to an external system) — the *one* place side effects are expected. |

Full, up-to-date reference: <https://spark.apache.org/docs/3.5.6/rdd-programming-guide.html>

### A subtlety: closures

Since RDD operations run on many separate machines, there is **no shared memory** between them. Any external variable your `map`/`filter`/etc. function refers to is captured in a **closure** and *copied* to every worker that needs it.

Two practical consequences:

1. **Keep closures small.** Referring to a huge local variable inside a `map` function means Spark has to ship a copy of it to every worker — this can silently kill performance.
2. **Closures are read-only from the worker's point of view.** If your function *mutates* a captured variable, that mutation happens on a copy living on the worker, and is **lost** — it is never sent back to the driver. If you need to communicate results back, you have to do it through the RDD itself (i.e. return values from the function), not through mutating captured state.


<a id="9"></a>
## 9. Lineage: narrow vs. wide dependencies

Every RDD remembers its **lineage** — the chain of parent RDDs and transformations used to build it. Not all dependencies in that chain are equally expensive, and the distinction drives most of Spark's scheduling decisions.

*(Figures below are from "Databricks Advanced Spark Training" by Reynold Xin, as cited in the original lecture.)*

![Narrow vs wide dependencies](images/narrow_wide_dependencies.png)

- **Narrow ("pipeline-able") dependencies** — each output partition depends on a **small, fixed number** of input partitions (often just one), e.g. `map`, `filter`, or a `union`. Crucially, a narrow-dependency partition can be **computed on the same physical machine** that already holds its input partition — no network traffic required. This is called **pipelining**.
- **Wide ("shuffle") dependencies** — an output partition may depend on **many / all** partitions of the parent RDD(s), e.g. `groupByKey` on non-partitioned data, or a `join` whose inputs aren't already co-partitioned the same way. This forces Spark to physically move data over the network between machines — the **shuffle**.

**Practical implication:** shuffles are the expensive part of a Spark job. Anything that needs a global reorganisation of data by key (grouping, joining unaligned data, sorting) will trigger one, and shuffles are simply **unavoidable** for such operations — but Spark tries hard to minimise how much data actually needs to cross the network (recall `reduceByKey`'s local pre-aggregation from Section 7).


<a id="10"></a>
## 10. DAG scheduling & pipelining into stages

Because transformations are lazy, Spark builds a full **DAG (Directed Acyclic Graph)** of RDD operations before running anything — this is what lets the `DAGScheduler` optimise the whole computation instead of executing each line of code eagerly, one at a time.

![Job scheduling process](images/dag_scheduling.png)

The scheduling process, in three steps:

1. **RDD objects** — your chain of `.join()`, `.groupBy()`, `.filter()`, `.count()` etc. calls builds up an *operator DAG* describing what should happen — but nothing runs yet.
2. **Scheduler (DAGScheduler)** — splits that DAG into **stages**, cutting the graph exactly at wide (shuffle) dependencies. Everything *within* a stage only has narrow dependencies, so it can be pipelined together with zero network traffic.
3. **Executors** — actually run the tasks (one task per partition, per stage) using worker threads, and manage a block manager to store and serve data blocks (e.g. cached RDDs, shuffle outputs) to other tasks.

### Pipelining into stages, visually

![Scheduler optimizations: pipelining into stages](images/pipelining_into_stages.png)

Reading this diagram:

- **Stage 1** does a `groupBy` — everything before that shuffle boundary (reading the data, any maps/filters) is pipelined together with zero network cost.
- **Stage 2** independently computes a `map` then a `union` on a different branch of the DAG — again all narrow dependencies, pipelined into one stage.
- **Stage 3** performs the final `join`, which is a wide dependency needing input from both Stage 1's and Stage 2's outputs.
- Grey boxes mark **previously computed partitions** — Spark's scheduler is smart enough to **reuse cached data** and skip recomputation where it can, instead of blindly re-running the whole DAG from scratch.
- The scheduler also **picks join algorithms based on existing partitioning**, specifically to minimise how much data needs to be shuffled.

**Why this matters in practice:** when you write idiomatic, chained Spark code (`.filter().map().groupBy().join()...`), you are *not* writing an imperative sequence of "do this, then do that" instructions the way you might in pandas. You're describing a *graph*, and Spark's scheduler decides the actual physical execution plan (what runs together, what needs a shuffle, what can be skipped because it's cached) — this is exactly analogous to how a SQL query optimiser works, and is not a coincidence: it's how **Spark SQL** itself was originally built, on top of these very same RDD primitives.


<a id="11"></a>
## 11. Spark ecosystem extensions

RDDs are Spark's foundational, low-level abstraction. On top of them, the Spark project ships several higher-level libraries:

| Extension | Purpose |
|---|---|
| **MLlib** | Distributed machine-learning library (classification, regression, clustering, recommendation, feature engineering, ML pipelines). |
| **Spark SQL** | Distributed analytics "SQL database" on top of RDDs — lets you query structured data with SQL or a DataFrame API, and benefits from a cost-based query optimiser. |
| **Spark Streaming / Structured Streaming** | Frameworks for processing unbounded, continuously-arriving data (as opposed to the fixed, static batch datasets we've used above). |
| **GraphX** | A distributed graph-processing system (PageRank-style algorithms, graph traversal, etc.), built on RDDs. |

The general pattern across all of them: keep the fault-tolerant, lineage-based RDD model underneath, and add a more convenient / more optimisable API layer on top for a specific problem domain.


<a id="12"></a>
## 12. Summary & further reading

**The one-paragraph version of this whole lecture:**

> Big datasets need to be processed by many machines in parallel, which introduces synchronization, load-balancing, and fault-tolerance problems. **MapReduce** solves this by restricting programmers to two pure functions, `Map` and `Reduce`, connected by a `Shuffle`-by-key step, so that any failed task can simply be re-executed elsewhere. **Apache Spark** keeps this same fault-tolerant, functional model — expressed through immutable, lineage-tracked **RDDs** — but adds in-memory caching between operations, a richer set of transformations, and a **DAG scheduler** that pipelines narrow-dependency operations together and only pays the network cost of a shuffle where a wide dependency genuinely requires it.

### Original sources referenced in this lecture

- J. Dean, S. Ghemawat — [*MapReduce: Simplified Data Processing on Large Clusters*](https://static.googleusercontent.com/media/research.google.com/en//archive/mapreduce-osdi04.pdf), OSDI 2004.
- M. Zaharia et al. — [*Resilient Distributed Datasets: A Fault-Tolerant Abstraction for In-Memory Cluster Computing*](https://www.usenix.org/system/files/conference/nsdi12/nsdi12-final138.pdf), NSDI 2012.
- J. Shi et al. — [*Clash of the Titans: MapReduce vs. Spark for Large Scale Data Analytics*](http://www.vldb.org/pvldb/vol8/p2110-shi.pdf), PVLDB 8(13), 2015.
- Yahoo! Developer Network — [Hadoop MapReduce Tutorial, Module 4](https://web.archive.org/web/20181215123514/https://developer.yahoo.com/hadoop/tutorial/module4.html).
- [Apache Hadoop MapReduce Tutorial](https://hadoop.apache.org/docs/current1/mapred_tutorial.html).
- [Apache Spark RDD Programming Guide](https://spark.apache.org/docs/3.5.6/rdd-programming-guide.html).
- Reynold Xin — *Databricks Advanced Spark Training* (source of the lineage/DAG-scheduling figures).
- Project Gutenberg — [*Kalevala*, the Finnish national epic](http://www.gutenberg.org/ebooks/7000) (used as the lecture's real-world word-count input).

### Suggested next steps

- Re-run Section 7 on the full text of the *Kalevala* (download it from the Project Gutenberg link above) instead of the two-line sample, and compare timings against the pure-Python version from Section 4.
- Try `groupByKey` instead of `reduceByKey` on a larger dataset and observe the difference in shuffle volume — this is a classic Spark performance pitfall.
- Explore the Spark UI (`http://localhost:4040` while a local job is running) to see the actual DAG, stages, and shuffle sizes for the word-count job above.
